# TIB final523: the application's summarizer family (Qwen2.5-7B-Instruct AWQ) on the 523 test IDs

Exploratory, post-hoc, text stage only (`protocol.json`, hash-checked first). One frozen prompt, greedy decoding,
no chunking, no fallback model, run **locally on this T4** (no API, no transcript leaves Colab). It cannot change
the final523 confirmatory label or the LED choice.

**Choose a T4 GPU runtime.** Inputs (private; TIB use is local/private/noncommercial), in Drive `MyDrive/TIB_FINAL523_APPLM_261031/`:
the code bundle `tib-final523-applm-code-261031-v1.zip`, the final523 scoring bundle `tib-final523-scoring-261024-v1.zip` and, for the dry run, the selection300
scoring bundle `tib-selection300-scoring-261024-v1.zip`.

**Order.** (1) `MODE = "dry_run"` first: 8 validation IDs; it prints counts, lengths and finish reasons, never text.
Only mechanical faults (template, token limits, stop tokens) may be fixed after it; the wording of the prompt may not.
(2) Set `MODE = "final"` and run again from the top. Every batch is checkpointed to Drive, so a dropped session is
resumed by running the notebook again. Budget ceiling 8 GPU hours (estimate 1.5-3).

In [1]:
import hashlib, json, platform, subprocess, sys, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/TIB_FINAL523_APPLM_261031")
    ROOT = Path("/content/applm"); ROOT.mkdir(exist_ok=True)
    with zipfile.ZipFile(DRIVE_DIR / "tib-final523-applm-code-261031-v1.zip") as bundle:
        bundle.extractall(ROOT)
    INPUT_DIR = DRIVE_DIR
    OUT_DIR = DRIVE_DIR / "results"
else:
    here = Path.cwd()
    ROOT = next(p for p in [here, *here.parents] if (p / "experiments/scripts/tib_final523_applm.py").exists())
    INPUT_DIR = ROOT / "research-data/tib-final523-scoring-261024-v1"
    OUT_DIR = ROOT / "research-data/tib-final523-applm-261031-v1"  # private: holds source and summary text
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))

PROTOCOL_PATH = ROOT / "experiments/registry/protocols/tib-final523-applm-261031-v1/protocol.json"
EXPECTED_PROTOCOL_SHA256 = "af8c9412070b0ad83ad677eabf1e527bbf06a647003e7f91ed32664e65f54820"
actual = hashlib.sha256(PROTOCOL_PATH.read_bytes()).hexdigest()
assert actual == EXPECTED_PROTOCOL_SHA256, (
    f"Protocol hash {actual} differs from the notebook's {EXPECTED_PROTOCOL_SHA256}: the code zip in Drive is "
    "probably an older build. Upload the latest tib-final523-applm-code-261031-v1.zip, restart the session and run again. Do not edit the protocol.")
print("protocol sha256 verified:", actual[:16], "...")
SCORING_ZIP = INPUT_DIR / "tib-final523-scoring-261024-v1.zip"
DRYRUN_ZIP = INPUT_DIR / "tib-selection300-scoring-261024-v1.zip"
MODE = "final"  # "dry_run": 8 validation IDs, lengths only | "final": the 523 test IDs
assert MODE in ("dry_run", "final")
SOURCE_ZIP = DRYRUN_ZIP if MODE == "dry_run" else SCORING_ZIP
assert SOURCE_ZIP.exists(), f"Missing input: {SOURCE_ZIP}"
SOURCE_ZIP_SHA256 = hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest()
OUTPUTS = OUT_DIR / ("PRIVATE_applm_dryrun.jsonl" if MODE == "dry_run" else "PRIVATE_applm_outputs.jsonl")
print("mode:", MODE, "| source bundle:", SOURCE_ZIP.name, SOURCE_ZIP_SHA256[:16], "...")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
protocol sha256 verified: af8c9412070b0ad8 ...
mode: final | source bundle: tib-final523-scoring-261024-v1.zip 6ae68b7a0ee95afe ...


In [2]:
import importlib.metadata as md_
def version(name):
    try:
        return md_.version(name)
    except md_.PackageNotFoundError:
        return None
ENVIRONMENT = {
    "python": platform.python_version(), "platform": platform.platform(),
    "packages": {n: version(n) for n in ["nltk", "scikit-learn", "scipy", "numpy", "pandas", "torch", "transformers"]},
    "code_sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
                    for p in sorted((ROOT / "experiments/scripts").glob("tib_*.py"))},
}
print(json.dumps(ENVIRONMENT["packages"], indent=1))

{
 "nltk": "3.9.1",
 "scikit-learn": "1.6.1",
 "scipy": "1.16.3",
 "numpy": "2.1.3",
 "pandas": "2.2.3",
 "torch": "2.8.0",
 "transformers": "4.57.1"
}


In [3]:
import subprocess
probe = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
print(probe.stdout.strip())
assert "T4" in probe.stdout, "Select a T4 GPU runtime"
ENVIRONMENT["gpu"] = probe.stdout.strip()

Tesla T4, 15360 MiB


## Install vLLM (pinned)
If `vllm==0.11.0` and `transformers==4.57.1` are not both installed, the cell installs them and stops: restart the runtime (Runtime > Restart session) and run again from the top.

In [4]:
import importlib.metadata as md_
def installed(name):
    try:
        return md_.version(name)
    except md_.PackageNotFoundError:
        return None
INSTALL_LOG = {}
PINS = dict(spec.split("==") for spec in ("vllm==0.11.0", "transformers==4.57.1"))
if any(installed(name) != version for name, version in PINS.items()):
    result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm==0.11.0", "transformers==4.57.1"], capture_output=True, text=True)
    INSTALL_LOG["vllm"] = {"returncode": result.returncode, "tail": (result.stderr or result.stdout)[-600:]}
    assert result.returncode == 0, INSTALL_LOG["vllm"]["tail"]
    raise SystemExit("vLLM and transformers installed. Restart the runtime (Runtime > Restart session) and run again from the top.")
ENVIRONMENT["packages"].update({n: installed(n) for n in ("vllm", "torch", "transformers")})
print({n: ENVIRONMENT["packages"][n] for n in ("vllm", "torch", "transformers")})

{'vllm': '0.11.0', 'torch': '2.8.0', 'transformers': '4.57.1'}


## Load the sources (hash-checked)

In [5]:
from experiments.scripts import tib_final523_applm as ap
SOURCES = ap.collect_sources(SOURCE_ZIP, limit=8 if MODE == "dry_run" else None)
# Fixed spread over strata and lengths, so the early time projection is not taken from the first sorted IDs.
SOURCES = [SOURCES[i] for i in sorted(range(len(SOURCES)), key=lambda i: (i * 7919) % len(SOURCES))]
print(len(SOURCES), "records; strata:", {s: sum(1 for x in SOURCES if x["stratum"] == s) for s in ap.STRATA})
assert MODE == "dry_run" or len(SOURCES) == 523, "the final run needs all 523 test IDs"

523 records; strata: {'short': 182, 'medium': 172, 'long': 169}


## Load the model (pinned revision)

In [6]:
import os
# Run the engine in this process: a crash of the separate engine process is invisible in Colab output,
# in-process it surfaces as an ordinary traceback. Same engine, same weights, same prompt.
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
from vllm import LLM, SamplingParams
LOAD_STARTED = time.monotonic()
llm = LLM(model="Qwen/Qwen2.5-7B-Instruct-AWQ", revision="b25037543e9394b818fdfca67ab2a00ecc7dd641", tokenizer_revision="b25037543e9394b818fdfca67ab2a00ecc7dd641", quantization="awq", dtype="float16",
          max_model_len=ap.MAX_MODEL_LEN, gpu_memory_utilization=0.80, max_num_batched_tokens=2048, seed=ap.SEED, max_num_seqs=4)
LOAD_SECONDS = time.monotonic() - LOAD_STARTED
tokenizer = llm.get_tokenizer()
SAMPLING = SamplingParams(temperature=0, top_p=1, max_tokens=ap.MAX_NEW_TOKENS, seed=ap.SEED)
print("model loaded in", round(LOAD_SECONDS), "s")

INFO 11-02 01:08:45 [__init__.py:216] Automatically detected platform cuda.
INFO 11-02 01:09:05 [utils.py:233] non-default args: {'dtype': 'float16', 'seed': 20261031, 'max_model_len': 32768, 'gpu_memory_utilization': 0.8, 'max_num_batched_tokens': 2048, 'max_num_seqs': 4, 'disable_log_stats': True, 'revision': 'b25037543e9394b818fdfca67ab2a00ecc7dd641', 'tokenizer_revision': 'b25037543e9394b818fdfca67ab2a00ecc7dd641', 'quantization': 'awq', 'model': 'Qwen/Qwen2.5-7B-Instruct-AWQ'}


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

INFO 11-02 01:09:29 [model.py:547] Resolved architecture: Qwen2ForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 11-02 01:09:29 [model.py:1510] Using max model len 32768
INFO 11-02 01:09:30 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=2048.


... (12 outputs left out)

[rank0]:W1102 01:11:37.423000 4133 torch/_inductor/utils.py:1436] [0/0] Not enough SMs to use max_autotune_gemm mode


INFO 11-02 01:12:01 [backends.py:218] Compiling a graph for dynamic shape takes 28.73 s
INFO 11-02 01:12:08 [monitor.py:34] torch.compile takes 37.57 s in total
INFO 11-02 01:12:10 [gpu_worker.py:298] Available KV cache memory: 5.95 GiB
INFO 11-02 01:12:11 [kv_cache_utils.py:1087] GPU KV cache size: 111,456 tokens
INFO 11-02 01:12:11 [kv_cache_utils.py:1091] Maximum concurrency for 32,768 tokens per request: 3.40x
WARNING 11-02 01:12:11 [gpu_model_runner.py:3663] CUDAGraphMode.FULL_AND_PIECEWISE is not supported with FlexAttentionMetadataBuilder backend (support: AttentionCGSupport.NEVER); setting cudagraph_mode=PIECEWISE because attention is compiled piecewise


Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 4/4 [00:01<00:00,  3.36it/s]


INFO 11-02 01:12:14 [gpu_model_runner.py:3480] Graph capturing finished in 3 secs, took 0.35 GiB
INFO 11-02 01:12:14 [core.py:210] init engine (profile, create kv cache, warmup model) took 51.62 seconds
INFO 11-02 01:12:15 [llm.py:306] Supported_tasks: ('generate',)
model loaded in 190 s


## Generate (checkpointed; run again to resume)
A prompt that cannot fit is a final `prompt_too_long` row and is never truncated. After the first five batches
(20 records) the cell projects the total time and stops if the projection exceeds the ceiling.

In [7]:
def generate_batch(batch_messages):
    ids = [list(tokenizer.apply_chat_template(m, add_generation_prompt=True, tokenize=True, return_dict=False))
           for m in batch_messages]
    outcomes, runnable = [None] * len(ids), []
    for i, prompt in enumerate(ids):
        if len(prompt) + ap.MAX_NEW_TOKENS > ap.MAX_MODEL_LEN:
            outcomes[i] = {"error": "prompt_too_long", "prompt_tokens": len(prompt)}
        else:
            runnable.append(i)
    if runnable:
        started = time.monotonic()
        try:
            results = llm.generate([{"prompt_token_ids": ids[i]} for i in runnable], SAMPLING, use_tqdm=False)
            share = (time.monotonic() - started) / len(runnable)
            for i, result in zip(runnable, results):
                out = result.outputs[0]
                outcomes[i] = {"text": out.text, "finish_reason": out.finish_reason, "prompt_tokens": len(ids[i]),
                               "completion_tokens": len(out.token_ids), "seconds": share}
        except Exception as exc:  # text-free reason; the batch is generated again on the next run
            import traceback
            # The dry run uses validation sources, so its traceback may be shown in full; the final run shows the type only.
            print(traceback.format_exc()[-2500:] if MODE == "dry_run" else "generation failed: " + type(exc).__name__)
            for i in runnable:
                outcomes[i] = {"error": type(exc).__name__, "prompt_tokens": len(ids[i])}
    return outcomes

CEILING = 39600
SPENT_PATH = OUT_DIR / "generation_seconds.json"
state = json.loads(SPENT_PATH.read_text()) if SPENT_PATH.exists() else {"seconds": 0.0, "batches": 0,
                                                                      "first_generate_utc": datetime.now(timezone.utc).isoformat()}
if MODE == "dry_run":
    state = {"seconds": 0.0, "batches": 0, "first_generate_utc": datetime.now(timezone.utc).isoformat()}
started, spent_before, batches_before = time.monotonic(), state["seconds"], state["batches"]

class CeilingReached(Exception):
    pass

def on_batch(rows, count=[0]):
    count[0] += 1
    state["seconds"] = spent_before + time.monotonic() - started
    state["batches"] = batches_before + count[0]
    if MODE == "final":
        SPENT_PATH.write_text(json.dumps(state))
    per_batch = state["seconds"] / state["batches"]
    if state["batches"] % 10 == 0:
        print(state["batches"], "batches,", round(state["seconds"] / 3600, 2), "GPU h")
    if state["batches"] == 5 and MODE == "final":
        projection = per_batch * -(-len(SOURCES) // 4)
        print("projected %.2f GPU h for all %d records" % (projection / 3600, len(SOURCES)))
        if projection > CEILING:
            raise CeilingReached(f"projected {projection / 3600:.2f} h exceeds the ceiling")
    if state["seconds"] + per_batch > CEILING:
        raise CeilingReached(f"the next batch would pass the ceiling at {state['seconds'] / 3600:.2f} h")

try:
    ROWS = ap.generate_rows(SOURCES, generate_batch, OUTPUTS, batch_size=4, on_batch=on_batch)
except CeilingReached as stop:
    raise SystemExit(f"{stop}. Every finished batch is checkpointed. Ask for more budget, then run again.")
ROWS = ap.latest_rows(OUTPUTS)  # a retried row appears twice in the file; the last line wins
RECEIPT = ap.generation_receipt(ROWS)
print(json.dumps(RECEIPT, indent=1))

110 batches, 8.63 GPU h
120 batches, 9.41 GPU h
130 batches, 10.1 GPU h
{
 "rows": 523,
 "scoreable": 523,
 "failed": 0,
 "failures": [],
 "status_counts": {
  "completed": 511,
  "completed_with_warning": 12,
  "failed": 0
 },
 "no_terminal_eos": 0,
 "repetition": 12,
 "forced_word_boundary_cut": 0,
 "delivered_words_mean": 127.95984703632887,
 "delivered_words_max": 303,
 "finish_reasons": {
  "stop": 523
 },
 "prompt_sha256": "757f5422cc58cc0252d272d53544131cd85a4dcb20b4c1f5d14b5f7dc972a9e4",
 "decoding": {
  "temperature": 0,
  "top_p": 1,
  "max_new_tokens": 600,
  "seed": 20261031,
  "repetition_penalty": 1
 },
 "prompt_tokens_max": 23077,
 "generation_seconds_total": 37125.47475091101
}


## Receipt (no text)

In [8]:
receipt = {"schema_version": 1, "protocol_id": ap.PROTOCOL_ID, "protocol_sha256": EXPECTED_PROTOCOL_SHA256, "mode": MODE,
           "source_bundle": SOURCE_ZIP.name, "source_bundle_sha256": SOURCE_ZIP_SHA256,
           "model": {"repo": "Qwen/Qwen2.5-7B-Instruct-AWQ", "revision": "b25037543e9394b818fdfca67ab2a00ecc7dd641"}, "environment": ENVIRONMENT,
           "model_load_seconds": LOAD_SECONDS, "generation_seconds_total": state["seconds"],
           "first_generate_utc": state["first_generate_utc"], "receipt_written_utc": datetime.now(timezone.utc).isoformat(),
           "summary": RECEIPT, "prompt_sha256": ap.PROMPT_SHA256, "decoding": ap.DECODING,
           "budget": {"protocol_ceiling_gpu_hours": 8, "approved_ceiling_gpu_hours": CEILING / 3600,
                      "note": "the run stopped at the protocol ceiling with 400 of 523 rows; the user approved a higher ceiling on 2026-11-02 to finish the remaining records with the same configuration"}}
path = OUT_DIR / ("applm_dryrun_receipt.json" if MODE == "dry_run" else "applm_generation_receipt.json")
path.write_text(json.dumps(receipt, indent=2), encoding="utf-8")
print(path, hashlib.sha256(path.read_bytes()).hexdigest())
if MODE == "final":
    assert RECEIPT["rows"] == 523, "rows differ from the 523 test IDs"

/content/drive/MyDrive/TIB_FINAL523_APPLM_261031/results/applm_generation_receipt.json f506f2e18c078c238c3185b5866cfab2f38e6bed0cb6d57e84ebc5e3e63d4ee2


## What to send back
`applm_generation_receipt.json` (counts, tokens, finish reasons, timing; no text) and the executed notebook.
`PRIVATE_applm_outputs.jsonl` holds sources, references and summaries: keep it in the private Drive folder and on
the local machine, never in the repository. The scoring notebook reads it from Drive.